# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features - this time from a merged cell CSV aready created
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Set your paths here

In [ ]:
# Imports
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# plotting
import plotly.express as px
import plotly.graph_objects as go

%matplotlib inline

import seaborn as sns

from mitolyso_plot_functions import *

from plate_preprocessing import *
from quality_control_functions import *
from single_csv_functions import (
    merge_ij_skeleton_features_into_combined_dataframe_from_folder,
)


## Import the big csv

In [ ]:
# import from a giant csv
# csvpath = "/Volumes/AllieS/Morphology_data/"
csvpath = "/Volumes/AllieS"
filename = "total_combined_cell_filtered_ij.csv"

# filename = "total_combined_cell_borders_excluded.csv"
final_filtered_df_with_ijskeleton = pd.read_csv(os.path.join(csvpath, filename))
display(final_filtered_df_with_ijskeleton.shape)

display(
    final_filtered_df_with_ijskeleton[
        final_filtered_df_with_ijskeleton["Metadata_PlateNumber"] == 4
    ][
        [
            "Metadata_PlateNumber",
            "Metadata_RowColFieldCode",
            "TotalMitochondria_Number_Object_Number",
            "TotalMitochondria_AreaShape_Area",
            "IJ_Mitochondria_Masks_MitochondrialFootprint"
        ]
        + search_column_name(final_filtered_df_with_ijskeleton, "IJ_Mitochondria")
    ].head(20)
)

### Remove problematic row/cols

### Filter out the poorly segmented cells based on the previously defined quality control filters and rename the columns

## Search Column Names

In [ ]:
# colnames
# sns.barplot(filter_df_2, x="AllGroups",y="AreaShape_Area", hue="PlateNumber", palette=colour_dict)
colnames = search_column_name(final_filtered_df_with_ijskeleton, "Plate")
display(final_filtered_df_with_ijskeleton[colnames])

In [ ]:
test_query_df = final_filtered_df_with_ijskeleton.query(
    "PlateNumber == 6 and Metadata_WellColumn == 2"
)

# display(test_query_df[[
#     "PlateNumber",
#     "Metadata_RowColFieldCode",
#     "Number_Object_Number",
#     "AllGroups",
#     "AreaShape_Area",
#     "Children_Mitochondria_Count",
#     "Children_Lysosomes_Count",
#     "Intensity_MeanIntensity_LAMP1_MAX",
#     "Intensity_MeanIntensity_MitoTracker_MAX",
# ]].head(20))
# display(
#     combined_cell_df_mitolyso.groupby(["PlateNumber", "Metadata_WellRow"])[
#         "Intensity_MeanIntensity_LAMP1_MAX"
#     ].describe()
# )


In [ ]:
def compare_features_barplots(
    df,
    feature1,
    feature2,
    group_col="PlateNumber",
    hue_col="Metadata_WellColumn",
    figsize=(12, 8),
):
    fig, ax = plt.subplots(2, 1, figsize=figsize, sharex=True)
    sns.barplot(
        data=df,
        ax=ax[0],
        x=group_col,
        y=feature1,
        hue=hue_col,
        palette="tab10",
        legend=True,
    )
    sns.barplot(
        data=df,
        ax=ax[1],
        x=group_col,
        y=feature2,
        hue=hue_col,
        palette="tab10",
        legend=False,
    )
    ax[0].legend(
        loc="upper right", bbox_to_anchor=(1.18, 1), borderaxespad=0, title=hue_col
    )
    plt.show()


compare_features_barplots(
    final_filtered_df_with_ijskeleton,
    "Intensity_MeanIntensity_LAMP1_MAX",
    "AreaShape_Area",
    hue_col="PlateNumber",
    group_col="Metadata_WellRow",
    figsize=(12, 8),
)
# fig,ax =plt.subplots(2, 1,figsize=(12, 6),sharex=True)
# sns.barplot(combined_cell_df_mitolyso, ax=ax[0], hue="Metadata_WellColumn", y="Children_Lysosomes_Count", x="PlateNumber", palette="tab10",legend=True)
# sns.barplot(combined_cell_df_mitolyso, ax=ax[1], hue="Metadata_WellColumn", y="Children_Mitochondria_Count", x="PlateNumber", palette="tab10",legend=False)
# ax[0].legend(loc="upper right", bbox_to_anchor=(1.1, 1), borderaxespad=0, title="Well Column")
# plt.show()


# Define the cell features


In [ ]:
#WIP Add extra columns
colour_dict = get_hard_code_plate_colours(final_filtered_df_with_ijskeleton)

def get_unique_cols_to_use(df):
    base_cols = [
        # "FileName_MitoTracker_MAX",
        "Metadata_PlateNumber",
        "Metadata_RowColFieldCode",
        "AllGroups",
        "AreaShape_Area",
    ]
    areashape_features = [
        "AreaShape_Area",
        "AreaShape_Perimeter",
        "AreaShape_EquivalentDiameter",
        "AreaShape_Eccentricity",
        "AreaShape_FormFactor",
        "AreaShape_Solidity",
        "AreaShape_Extent",
        "AreaShape_MaxFeretDiameter",
        "AreaShape_MinFeretDiameter",
        "AreaShape_MeanRadius",
    ]

    colnames_mitoskel_nuc = search_column_name(df, "Nuclei_ObjectSkeleton")
    colnames_mitocount = search_column_name(
        df, ["Children_Mitochondria", "Count"], inclusive_or=False
    )
    colnames_mitoarea = search_column_name(
        df, ["Mito", "AreaShape_Area"], inclusive_or=False
    )

    colnames_lysocount = search_column_name(
        df, ["Children_Lysosomes", "Count"], inclusive_or=False
    )
    colnames_lysoarea = search_column_name(
        df, areashape_features, inclusive_or=True
    )
    for col in colnames_lysoarea.copy():
        if "Lyso" in col and "AreaShape" in col:
            continue
        else:
            colnames_lysoarea.remove(col)

    colnames_intesnity_distribution = search_column_name(
        df, ["Radial","MitoTracker"], inclusive_or=False
    )
    colnames_intesnity_distribution += search_column_name(
        df, ["Radial","LAMP1"], inclusive_or=False
    )
    for col in colnames_intesnity_distribution.copy():
        if "Nuclei" in col or "Closing" in col:
            colnames_intesnity_distribution.remove(col)
        else:
            continue
        
    colnames_overlap = search_column_name(df, ["Overlap","Correlation"], inclusive_or=True)
    for col in colnames_overlap.copy():
        if ("Mito" not in col and "Lyso" not in col and "LAMP1" not in col) or  ("Texture" in col or "DAPI" in col):
            colnames_overlap.remove(col)
        else:
            continue
    print(colnames_overlap)
    
    colnames_ij = search_column_name(df, "IJ_Mitochondria")

    use_cols = (
        base_cols
        + colnames_mitoskel_nuc
        + colnames_ij
        + colnames_mitocount
        + colnames_mitoarea
        + colnames_lysocount
        + colnames_lysoarea
        + colnames_overlap
        + colnames_intesnity_distribution
    )
    use_cols_unique = list(dict.fromkeys(use_cols))
    # scrub out any non-numeric columns that we aren't going to use for analysis
    use_cols_unique_copy = use_cols_unique.copy()
    for col in use_cols_unique_copy:
        if col in base_cols:
            continue
        elif "Metadata" in col or "Title" in col or "FileName" in col:
            use_cols_unique.remove(col)
    return use_cols_unique


def get_object_skeleton_length_cols(df):
    colnames_object_skeleton_length = search_column_name(df, "SkeletonLength")
    for col in colnames_object_skeleton_length.copy():
        if "Mean" in col or "Median" in col or "Threshold" in col or "Stdev" in col or "FromBranches" in col:
            colnames_object_skeleton_length.remove(col)

    print(colnames_object_skeleton_length)
    return colnames_object_skeleton_length


skeleton_length_cols = get_object_skeleton_length_cols(final_filtered_df_with_ijskeleton)
print(skeleton_length_cols)
cols_to_use = get_unique_cols_to_use(final_filtered_df_with_ijskeleton)

In [ ]:
def make_per_cell_area_column_names(
    df,
    use_cols,
    area_col="AreaShape_Area",
    colnames_mitoskel_seeds=None,
    number_of_seeds_col="Children_MitoSkel_Seeds_Count",
    areashape_cols=None,
    calculate_totals=False,
    base_cols=None,
    exclude_per_area=None
):
    if base_cols is None:
        base_cols = ["Metadata_PlateNumber", "Metadata_RowColField", "AllGroups", "AreaShape_Area"]
    if areashape_cols is None:
        areashape_cols = ["AreaShape_Area", "AreaShape_Perimeter", "AreaShape_EquivalentDiameter"]
    df = df.copy()
    new_use_cols = use_cols.copy()
    for col in base_cols:
        if col in new_use_cols:
            new_use_cols.remove(col)
    
    new_columns = {}
       
    count_flag = 0
    for col in areashape_cols:
        #remove the col from new_use_cols so that it doesn't get processed again in the final loop  
        if col in new_use_cols:
            new_use_cols.remove(col)
            
        df[col] = pd.to_numeric(df[col], errors="coerce")
        
        if calculate_totals and "Mean" in col:
            #calculate the total area occupied and then divide by area
            col_without_mean = col.replace("Mean_", "")
            new_columns["Math_Total_" + col_without_mean] = (
                df[col] * df[count_cols[count_flag]]
            )
            new_columns["Per_Area_AreaOccupied_" + col_without_mean] = (
                df["Math_Total_" + col_without_mean] / df[area_col]
            )

        elif "Mean" in col:
            continue
        elif "Total" in col:
            col_without_total = col.replace("Total_", "")
            new_columns["Per_Area_AreaOccupied_" + col_without_total] = df[col] / df[area_col]
        elif "RelabeledMito" in col:
            new_columns["Per_Area_AreaOccupied_" + col] = df[col] / df[area_col]
        else:
            new_columns["Per_Area_" + col] = df[col] / df[area_col]
               
    # calculate total per cell for mito skel seeded version (if using), then divide by area
    if colnames_mitoskel_seeds:
        for col in colnames_mitoskel_seeds:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            col_without_mean = col.replace("Mean_", "")
            new_columns["Math_Total_" + col_without_mean] = df[col] * df[number_of_seeds_col]
            new_columns["Per_Area_" + col_without_mean] = new_columns["Math_Total_" + col_without_mean] / df[area_col]
            if col in new_use_cols:
                new_use_cols.remove(col)

    #for everything else in use_cols that makes sense, just divide by area
    if exclude_per_area is None:
        exclude_per_area = [
            "Mean",
            "Median",
            "Std",
            "Distribution",
            "Metadata",
            "Title",
            "Per_Area",
            "Location",
            "Correlation",
            "Overlap",
            "Eccentricity",
            "FormFactor",
            "Solidity",
            "Extent",
            "BranchLength"
        ]
    for col in new_use_cols:
        if any(exclude in col for exclude in exclude_per_area):
            continue
        else:
            print("making per area column for:", col)
            df[col] = pd.to_numeric(df[col], errors="coerce")
            #make new colname by adding per area
            new_colname = "Per_Area_" + col 
            if "Count" in col:
                #make new colname by replacing Children_ with Number_ for readbility
                new_colname = new_colname.replace("Children_", "Number_")
                
                
            new_columns[new_colname] = df[col] / df[area_col]
        
    new_columns_df = pd.DataFrame(new_columns, index=df.index)
    df = pd.concat([df, new_columns_df], axis=1)   
    return df

def make_per_skeleton_length_column_names(
    df,
    use_cols,
    skeleton_length_cols,
    base_cols=None,
    colnames_mitoskel_seeds=None,
    sets=None,
    feature_types=None,
):
    if sets is None:
        sets = []
    if feature_types is None:
        feature_types = ["Nuclei_ObjectSkeleton", "MitoSkel_Seeds_ObjectSkeleton", "IJ_Mitochondria"]
    df = df.copy()
    new_use_cols = use_cols.copy()
    if base_cols is None:
        base_cols = [
            "Metadata_PlateNumber",
            "Metadata_RowColField",
            "AllGroups",
            "AreaShape_Area",
        ]   
    # take out things that don't make sense
    for col in use_cols:
        if (
            col in base_cols
            or col in skeleton_length_cols
            or "Metadata" in col
            or "Title" in col
            or "Mean" in col
            or "Median" in col
            or "Stdev" in col
            or "Footprint" in col
            or "Per_Area" in col
            or "Correlation" in col
            or "Overlap" in col
            or "Distribution" in col
            or "AreaShape" in col
            or "Children" in col
        ):
            new_use_cols.remove(col)

    new_columns = {}

    for col in new_use_cols:
        # check if the column is one of the feature types we want to process
        this_feature_type = None
        for feature_type in feature_types:
            if feature_type in col:
                this_feature_type = feature_type
                break

        df[col] = pd.to_numeric(df[col], errors="coerce")

        for length_col in skeleton_length_cols:
            # only divide by the skeleton length column that matches the feature type of the current column
            if this_feature_type and this_feature_type not in length_col:
                continue
            elif this_feature_type == "IJ_Mitochondria":
                subtypes = ["_LargestStructure", "_TotalAcrossAllStructures"]
                if any(subtype in col for subtype in subtypes) and any(
                    (subtype in col and subtype not in length_col)
                    or (subtype not in col and subtype in length_col)
                    for subtype in subtypes
                ):
                    continue
            print(f"Calculating Per_SkeletonLength for {col} using {length_col}")
            if sets and any(set_name in length_col for set_name in sets):
                for set_name in sets:
                    if set_name in length_col and set_name in col:
                        print(f"Using set {set_name} for {col} and {length_col}")
                        new_columns["Per_SkeletonLength_" + col] = df[col] / df[length_col]
                        break
            else:
                new_columns["Per_SkeletonLength_" + col] = df[col] / df[length_col]

    new_columns_df = pd.DataFrame(new_columns, index=df.index)
    df = pd.concat([df, new_columns_df], axis=1)    
    return df

extrafeatures_filtered_cell_df_mitolyso = make_per_cell_area_column_names(
    final_filtered_df_with_ijskeleton,
    use_cols=cols_to_use,
)
display(extrafeatures_filtered_cell_df_mitolyso.head())
extrafeatures_filtered_cell_df_mitolyso = make_per_skeleton_length_column_names(
    extrafeatures_filtered_cell_df_mitolyso,
    use_cols=cols_to_use,
    skeleton_length_cols=skeleton_length_cols,
    feature_types=[
        "Nuclei_ObjectSkeleton",
        "MitoSkel_Seeds_ObjectSkeleton",
        "IJ_Mitochondria",
    ],
)

In [ ]:
def mean_intensity_per_compartment_per_cell(df, compartment, name, tag, math=None):
    # Calculate the mean intensity of each compartment per cell
    # mean_intesity_per_compartment = integrated / (children*mean_area)
    colname = f"Mean_Intensity_Per_{compartment} Per_Cell"
    integrated = "Intensity_IntegratedIntensity_" + tag + "_MAX"
    # children = 'Children_' + compartment + '_Count'
    # mean_area = 'Mean_'+ compartment + '_AreaShape_Area'
    total_organelle_area = name + "_AreaShape_Area"
    total_organelle_area = math if math is not None else total_organelle_area

    df[colname] = df.apply(lambda x: x[integrated] / x[total_organelle_area], axis=1)

    return df[colname]


# Calculate the total area occupied by mitochondria and lysosomes per cell
def calculate_extra_features(full_df, organelles=["Mitochondria", "Lysosomes"]):
    """_summary_

    Args:
        full_df (DataFrame): _description_

    Returns:
        df (DataFrame): the df with all the feature calcs
    """
    df = full_df.copy()
    # Total intensity per cell based on integrated instensity if I don't already have the merged area
    for organelle in organelles:
        if organelle == "Mitochondria":
            tag = "MitoTracker"
            name = "TotalMitochondria"
            math = None
        elif organelle == "Lysosomes":
            tag = "LAMP1"
            name = "TotalLysosomes"
            math = None
        else:
            continue
        df[f"Mean_Intensity_Per_{organelle}_PerCell_Area"] = (
            mean_intensity_per_compartment_per_cell(
                df, organelle, name, tag, math=math
            )
        )
        # Compartment diameter ratios
        df[f"Ratio_Mean_{organelle}_MaxMinFeret_DiameterRatio"] = (
            df[f"Mean_{organelle}_AreaShape_MaxFeretDiameter"]
            / df[f"Mean_{organelle}_AreaShape_MinFeretDiameter"]
        )
        # organelle is also prefix here
        df[f"Ratio_Median_{organelle}_DiameterRatio_PerCell"] = (
            df[f"{organelle}_Median_{organelle}_AreaShape_MaxFeretDiameter"]
            / df[f"{organelle}_Median_{organelle}_AreaShape_MinFeretDiameter"]
        )

        # Quin's Ratio: Ratio of centroid distance to minimum distance for mitochondria and lysosomes
        # increase = more peripheral, decrease = more nuclear
        df[f"Ratio_Mean_{organelle}_Distance_Centroid_Cell_Minimum_Cell_QuinRatio"] = (
                df[f"Mean_{organelle}_Distance_Centroid_Cell"]
                / df[f"Mean_{organelle}_Distance_Minimum_Cell"]
            )

        # Distance to parents percellarea
        df[f"Per_Area_Mean_{organelle}_Distance_Centroid_Cell"] = (
            df[f"Mean_{organelle}_Distance_Centroid_Cell"] / df["AreaShape_Area"]
        )
        df[f"Per_Area_Mean_{organelle}_Distance_Minimum_Cell"] = (
            df[f"Mean_{organelle}_Distance_Minimum_Cell"] / df["AreaShape_Area"]
        )
        
    # mitolyso related
    df["Ratio_Number_Lysosomes_To_Mitochondria"] = (
        df["Children_Lysosomes_Count"] / df["Children_Mitochondria_Count"]
    )
    # df["Density_Lysosomes_Mitochondria_Ratio"] = (
    #     df["OccupiedAreaFraction_Lysosomes_PerCell_Area"]
    #     / df["OccupiedAreaFraction_Mitochondria_PerCell_Area"]
    # )
    df["Ratio_Area_Lysosomes_To_Mitochondria"] = (
        df["TotalLysosomes_AreaShape_Area"]
        / df["TotalMitochondria_AreaShape_Area"]
    )

    return df


extrafeatures_filtered_cell_df_mitolyso = calculate_extra_features(
    extrafeatures_filtered_cell_df_mitolyso
)
# display(extrafeatures_filtered_cell_df_mitolyso)


### Make a convenient table for display and/or plots

In [ ]:
def make_display_df_and_pivot_tables(df, use_cols=None, make_new_features=False, display_table=True):
    if use_cols is None:
        use_cols = get_unique_cols_to_use(df)
    if make_new_features:
        df = calculate_extra_features(df)
        df = make_per_cell_area_column_names(
            df, use_cols=use_cols
        )
        skeleton_length_cols = get_object_skeleton_length_cols(df)
        df = make_per_skeleton_length_column_names(
            df,
            use_cols=use_cols,
            skeleton_length_cols=skeleton_length_cols,
            feature_types=[
                "Nuclei_ObjectSkeleton",
                "MitoSkel_Seeds_ObjectSkeleton",
                "IJ_Mitochondria",
            ],
        )
    colnames_per_area = search_column_name(df, "Per_Area")
    colnames_per_skeletonlength = search_column_name(df, "Per_SkeletonLength")
    use_cols_new = use_cols + colnames_per_area + colnames_per_skeletonlength
    use_cols_new_unique =list(dict.fromkeys(use_cols_new))
    
  
    display_df = df[use_cols_new_unique]
    display_df_pivot_plates = display_df.pivot_table(
        index=["Metadata_PlateNumber", "AllGroups"],
        values=use_cols_new_unique[3:],
        #columns=["AllGroups"],
        aggfunc="mean",
    )
    if display_table:
        display(display_df)
        display(display_df_pivot_plates)
    return display_df, display_df_pivot_plates

# display_df, display_df_pivot_plates = make_display_df_and_pivot_tables(
#     extrafeatures_filtered_cell_df_mitolyso,
#     use_cols=cols_to_use,
#     make_new_features=False,
#     display_table=True
#     )



### Make feature dictionaries

In [ ]:
def get_feature_dicts(df):
    columns_list = define_cell_features(df)
    mito_features = make_feature_dict(
        [
            col
            for col in columns_list
            if ("Mito" in col or "Mitochondria" in col)
            and ("DAPI" not in col and "LAMP1" not in col and "Frame" not in col)
            and not (col.startswith("Nuclei_"))
        ]
    )
    lyso_features = make_feature_dict(
        [
            col
            for col in columns_list
            if ("Lysosome" in col or "LAMP1" in col or "Lyso" in col)
            and ("DAPI" not in col and "Mito" not in col and "Frame" not in col)
            and not (col.startswith("Nuclei_"))
        ]
    )
    nuc_features = make_feature_dict(
        [
            col
            for col in columns_list
            if ("Nuc" in col or "DAPI" in col)
            and ("MitoTracker" not in col and "LAMP1" not in col and "Frame" not in col)
        ]
    )
    cell_features = make_feature_dict(
        [
            col
            for col in columns_list
            if "AreaShape" in col
            and "Mito" not in col
            and "Lyso" not in col
            and "LAMP1" not in col
            and "Nuc" not in col
            and "DAPI" not in col
            and "Metadata" not in col
            and "FileName" not in col
            and "PathName" not in col
        ]
    )
    return [mito_features, lyso_features, nuc_features, cell_features]

## Feature lists here:

In [ ]:
feature_dicts = get_feature_dicts(extrafeatures_filtered_cell_df_mitolyso)
feature_names = [
    "Mitochondria Features",
    "Lysosome Features",
    "Nucleus Features",
    "Cell Features",
]

# Define the output file path
output_file_path = "allfeatures_file.md"

# Open the file in write mode
with open(output_file_path, "w") as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f'"{feature}",\n')
            file.write("\n")

print(f"List has been written to {output_file_path}")


In [ ]:
test_df = extrafeatures_filtered_cell_df_mitolyso
pivot_df = test_df.pivot_table(
    index=["Metadata_PlateNumber", "AllGroups"],
    values=["AreaShape_Area"],
    aggfunc="mean",
)
display(pivot_df)

## Check if there are any mixed types hiding out

In [ ]:
# # Debug to check for mixed types in columns
# for col in extrafeatures_filtered_cell_df_mitolyso.columns:
#     types = extrafeatures_filtered_cell_df_mitolyso[col].apply(type).value_counts()
#     if len(types) > 1:
#         print(f"Column '{col}' has mixed types: {types}")

# Functions for Data Analysis
calulcate normalizations, remove extreme left outliers, etc

## Normalize features to control (Passage 6-8)

In [ ]:
# updated/faster version that doesn't use apply and is much faster for large dataframes
valid_feature_cols = get_valid_numeric_features(
    extrafeatures_filtered_cell_df_mitolyso, feature_dicts
)

group_col = "AgeGroup"
control_value = 0

norm_combined_cell_df_mitolyso = normalize_quantities_to_control_group_average(
    extrafeatures_filtered_cell_df_mitolyso.copy(),
    valid_feature_cols,
    group_col,
    control_value,
    overwrite=True,
    drop_avg_cols=True,
)


# It's plotting time


## Most interesting features so far
- Intensity_MassDisplacment_MitoTracker - increase
- Median Mitochondria Location CenterMass Intensity X
  - decrease,splits into bimodal
  - similar for y
  - AreaShape_Center_X and Center_Y also have similar pattern
  - And Location_Center
- Median mitochondria loaction max intensity
  - Same as CenterMass
- Mean Mitochondria Solidity - increase to p29
- Children Mitochondria Count - gradual increase
  - But scales with size - Density is not sig (slight increase)
  - Total mito area increases; complements this
- Mean Mito Centroid distance - increased (more peripheral)
  - But median is much more vairable between batches
- Mean Mito Trunks - decreased
- Median mito area per cell area - increased
### Lysosomes
- Total area also goes up
- rep3 looks like an outlier here
- Also has the location_ maxintensity change and go bimodal
- bimodal-looking intensity
### Nuclei
- Nuc area increases; scales with cell size increases
- solidity, extent down
- Nuc area ratio - slight increase


## Distribution plots

In [ ]:
# Functions to visualize the distributions
# Define and use a simple function to label the plot in axes coordinates
def ridge_label(x, color, label):
    ax = plt.gca()
    ax.text(
        -0.1,
        -0.2,
        label,
        fontweight="bold",
        color=color,
        ha="left",
        va="center",
        transform=ax.transAxes,
    )


def seaborn_ridgeplot(
    df,
    value_col,
    group_col,
    palette=None,
    bw_adjust=1,
    xlabel=None,
    xlim=(None, None),
    title=None,
    fill_alpha=1,
    linewidth=1.5,
    figsize=(20, 30),
    save=True,
    out_dir="",
    show_percentiles=True,
    truncate_outliers=True,
    norm=False,
):
    """
    Make a ridgeline (joyplot) using seaborn FacetGrid and kdeplot
    Args:
        df: DataFrame
        value_col: str, column with numeric values
        group_col: str, column with group/category
        palette: seaborn palette or list/dict of colors
        bw_adjust: float, KDE bandwidth adjust
        xlabel: str or None
        title: str or None
        fill_alpha: float, alpha for fill
        linewidth: float, line width for outline
        figsize: tuple, figure size
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

    unique_groups = df[group_col].unique()
    if palette is None:
        palette = sns.cubehelix_palette(len(unique_groups), rot=-0.25, light=0.7)
    else:
        palette = palette

    if truncate_outliers and xlim == (None, None):
        try:
            if norm:
                top_fence = df[value_col].mean() + 10 * df[value_col].std()
                bottom_fence = None  # np.percentile(data_df[y_value], 0.000001)
            else:
                top_fence = np.percentile(df[value_col], 99.9)
                bottom_fence = np.percentile(df[value_col], 0.01)
                # handle errors where the data is very skewed and the percentile is inf or nan
                if top_fence == 0 or np.isnan(top_fence) or np.isinf(top_fence):
                    top_fence = None
                if (
                    bottom_fence == 0
                    or np.isnan(bottom_fence)
                    or np.isinf(bottom_fence)
                ):
                    bottom_fence = None
            xlim = (bottom_fence, top_fence)
        except ValueError as e:
            print(e)
            top_fence = None
            bottom_fence = None
            xlim = (None, None)
        print(f"Truncating outliers at: {xlim}")
        xmin, xmax = xlim
        plot_df = df.copy()
        if xmin is not None:
            plot_df = plot_df[plot_df[value_col] >= xmin]
        if xmax is not None:
            plot_df = plot_df[plot_df[value_col] <= xmax]
    else:
        plot_df = df.copy()
    # Initialize the FacetGrid object
    g = sns.FacetGrid(
        plot_df,
        row=group_col,
        hue=group_col,
        aspect=15,
        height=0.5,
        palette=palette,
        xlim=xlim,
    )
    # Draw the densities in a few steps
    g.map(
        sns.kdeplot,
        value_col,
        bw_adjust=bw_adjust,
        clip_on=False,
        fill=True,
        alpha=fill_alpha,
        linewidth=linewidth,
    )
    g.map(sns.kdeplot, value_col, clip_on=False, color="w", lw=2, bw_adjust=bw_adjust)

    if show_percentiles:
        percentiles = [5, 12.5, 25, 50, 75, 87.5, 95]
        for ax in g.axes.flatten():
            # group label text (FacetGrid puts "group_col = <value>" in the title)
            title_text = ax.get_title()
            if " = " in title_text:
                group_val = title_text.split(" = ", 1)[1]
            else:
                group_val = title_text

            group_data = df[df[group_col] == group_val][value_col].dropna()
            if group_data.empty:
                continue

            # pick the most representative line on the axis (the KDE line)
            lines = ax.get_lines()
            if not lines:
                continue
            # choose the line with the largest x-range (robust against multiple lines)
            kde_line = max(lines, key=lambda l: np.ptp(l.get_xdata()))
            xs = kde_line.get_xdata()
            ys = kde_line.get_ydata()
            # line_colour = kde_line.get
            # compute median and interpolate its KDE height
            median = np.median(group_data)
            height = np.interp(median, xs, ys, left=0.0, right=0.0)

            # draw a solid thicker median line from y=0 up to the KDE height
            ax.vlines(
                median, 0, height, color="black", linewidth=3, linestyle=":", zorder=4
            )
            ax.set_xlim(xlim)
            # draw lighter dashed lines for the percentiles (optional)
            group_percentiles = np.percentile(group_data, percentiles)
            for p in group_percentiles:
                ax.vlines(p, 0, height, color="dimgray", ls=":", alpha=0.6, linewidth=2)
    # passing color=None to refline() uses the hue mapping
    g.refline(y=0, linewidth=2, linestyle="-", color=None, clip_on=False)

    g.map(ridge_label, value_col)

    # Set the subplots to overlap
    g.figure.subplots_adjust(hspace=-0.25)
    g.figure.set_size_inches(figsize)
    # Remove axes details that don't play well with overlap
    g.set_titles("")
    g.set(yticks=[], ylabel="", xlim=xlim)
    g.despine(bottom=True, left=True)
    if xlabel:
        plt.xlabel(xlabel, fontweight="bold", fontsize=14)
    else:
        plt.xlabel(value_col, fontweight="bold", fontsize=14)
    if title:
        g.figure.suptitle(title, ha="right", fontsize=18, fontweight="bold")
    plt.xlim(xlim)
    plt.tight_layout()
    if save:
        plt.savefig(f"{Path(out_dir, f'{value_col}_{group_col}_joyplot')}.png")
    plt.show()


def plotly_histogram(df, y_value, group_var, save=False, out_dir=""):
    import kaleido

    df_sorted = df.sort_values(
        by=[group_var], key=lambda x: x.map(passage_groups_sort_key)
    ).reset_index(drop=True)
    hist2 = px.histogram(
        df_sorted,
        x=y_value,
        color=group_var,
        marginal="box",
        # histnorm='probability density',
        # range_x=(0, top_fence),
    )
    hist2.write_image(Path(out_dir, f"{y_value}_histogram.png"), scale=1.5)
    hist2.show()


## Helper functions for plot building

In [ ]:
def annotate_pairs_with_calculated_pvalues(
    ax,
    data,
    pivot_data,
    x_value,
    y_value,
    plate_col_name="Plate_Name",
    test_name="tukey",
    pairs=None,
    order=None,
    plot_type="violinplot",
    show_test_name=False,
    p_correction="fdr_bh",
    annotation_location="inside",
    line_offset=0.0,
    line_offset_to_group=0.005,
    text_offset=0.15,
    line_height=0.05,
):
    """Add statistical annotations to a plot using a multiple comparisons test in the statsmodels, scipy, or scikit-posthocs modules.
    see https://statannotations.readthedocs.io/en/latest/custom-test.html for more examples
    Also see https://www.graphpad.com/guides/prism/latest/statistics/stat_summary_of_multiple_comparison.htm for a list of posthoc tests and when to use them

    Args:
        ax (Matplotlib Axes Object): the axis of the graph to annoate
        data (DataFrame): dataframe from a grouped feature df containing the groups aggregated by plate to analyze in "tidy" format
        pivot_data (DataFrame): the grouped feature df in matrix format
        x_value (str): independent variable on x axis
        y_value (str): dependent variable on y axis
        plate_col_name (str, optional): the col containing the experimental plate. Defaults to "Plate_Name".
        test_name (str, optional): the statistical test to perform. Accepts values of "tukey", "anova", or "tukey_v2", for ANOVA with Tukey's HSD, "tukey_v3" for ANOVA with Tukey HSD with Tukey-Kramer correction, "games-howell" or "games" for ANOVA with Games-Howell posthoc, "rmanova" for repeated-measures ANOVA using Welch's ttest with the specified p-value correction, "kruskal" or "dunn" for classic nonparametric multiple comparisons with Dunn's postc, "conover" for kruskal with Conover's posthoc, "nemenyi" for kruskal (or friedman) with Nemenyi's posthoc for repeated measures, "pairwise_ttest" for corrected ttests, "pairwise_mwu" for nonparametic multiple comparisons. Defaults to "tukey".
        pairs (list of str, optional): The pairs of x_value for the comparisons. Defaults to None, is automatically calculated otherwise based on the getpairs() function.
        order (listlike, optional): _description_. Defaults to None.
        plot (str, optional): the type of plot to annotate. Defaults to "violinplot".
        p_correction (str, optional): the p-value correction to use if applicable. Deaults to Benjamini/Hochberg "fdr_bh" (non-negative) method ; graphpad reccomneds as its less hemmoraging to your power. Also accepts "holm", "sidak", "bonferroni", "holm-sidak" and Benjamini/Yekutieli "fdr-by" for negative values. See https://scikit-posthocs.readthedocs.io/en/latest/generated/scikit_posthocs.posthoc_mannwhitney.html for other options

    Returns:
        _type_: _description_
    """
    from statannotations.Annotator import Annotator
    print(f"Annotating with statistical test: {test_name}")

    if pairs is None:
        pairs = getpairs(data, x_value, order=order)
    # nonparametric tests
    if test_name in ["kruskal", "dunn", "kruskal-wallis"]:
        used_pairs, p_values = kruskal_with_dunn_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            p_correction=p_correction,
            display_results=True,
        )
    elif test_name in ["drubin", "drubin_posthoc"]:
        used_pairs, p_values = kruskal_with_drubin_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            p_correction=p_correction,
            display_results=True,
        )
    elif test_name in ["conover", "con", "kruskal-conover"]:
        used_pairs, p_values = kruskal_with_conover_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            p_correction=p_correction,
            display_results=True,
        )
    elif test_name in ["nemenyi", "kruskal-nemenyi"]:
        used_pairs, p_values = kruskal_with_nemenyi_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            display_results=True,
            p_correction=p_correction,
        )
    # parametric tests
    elif test_name in ["anova", "tukey", "tukeyhsd"]:
        # perform anova and tukey's post-hoc test
        used_pairs, p_values = pvalues_anova_and_tukeyhsd_posthoc(
            data, pivot_data, x_value, y_value, order=order, desired_pairs=pairs
        )
    elif test_name in ["games", "games-howell"]:
        used_pairs, p_values = pvalues_anova_with_games_howell_pingouin(
            data,
            pivot_data,
            x_value,
            y_value,
            order=order,
            desired_pairs=pairs,
            display=True,
        )
    elif test_name in ["tahmane", "tahmane-t2"]:
        used_pairs, p_values = anova_with_tahmane_posthoc(
            data,
            x_value,
            y_value,
            order=order,
            desired_pairs=pairs,
            display_results=True,
        )
    elif test_name in ["tukey_v2", "tukey_posthocs"]:
        used_pairs, p_values = anova_with_tukey_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            plate_number_col=plate_col_name,
            order=order,
            desired_pairs=pairs,
            display_results=True,
        )
    elif test_name in ["tukey_v3", "tukey_pingouin"]:
        used_pairs, p_values = pvalues_anova_with_tukey_pingouin(
            data,
            x_value=x_value,
            y_value=y_value,
            display=True,
        )
    elif test_name in ["pairwise_ttest" or "multiple_ttest"]:
        used_pairs, p_values = pvalues_anova_with_pairwise_tests_pingouin(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            pval_correction=p_correction,
            parametric=True,
            display=True,
        )
    elif test_name in [
        "pairwise_mwu"
        or "multiple_mwu"
        or "pairwise_mannwhitney"
        or "multiple_mannwhitney"
        or "pairwise_wilcoxon"
        or "multiple_wilcoxon"
    ]:
        used_pairs, p_values = pvalues_anova_with_pairwise_tests_pingouin(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            pval_correction=p_correction,
            parametric=False,
            display=True,
        )
    elif test_name in ["ttest_posthoc", "welch's_posthoc", "tt_posthoc"]:
        used_pairs, p_values = anova_with_corr_ttest_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            order=order,
            desired_pairs=pairs,
            p_corr=p_correction,
            display_results=True,
        )
    else:
        raise ValueError(
            f"Test name '{test_name}' is invalid. Use 'tukey', 'anova', 'kruskal', 'games-howell', 'drubin', 'tukey_v2', or 'dunn'."
        )
    if used_pairs is None or len(used_pairs) == 0:
        print(f"No significant pairs found for the {test_name} test.")
        return ax
    else:
        annotator = Annotator(
            ax=ax,
            pairs=list(used_pairs),
            data=data,
            plot=plot_type,
            x=x_value,
            y=y_value,
            order=order,
        )
        annotator.reset_configuration()
        # see https://raw.githubusercontent.com/trevismd/statannotations/3f020ae631ca88a091b6ee3e9a9fd32158920879/usage/example_tuning_y_offsets_w_arguments.png
        annotator.configure(
            text_format="full",
            test_short_name=test_name,
            pvalue_format_string="{:.4f}",
            fontsize="small",
            # pvalue_format = [[1e-5, "1e-5"], [1e-4, "1e-4"], [1e-3, "0.001"], [1e-2, "0.01"], [5e-2, "0.05"]],
            loc=annotation_location,
            hide_non_significant=True,
            color="black",
            verbose=2,
            line_height=line_height,
            text_offset=text_offset,
            line_offset_to_group=line_offset_to_group,
            line_offset=line_offset,
            show_test_name=show_test_name,
        )
        annotator.set_pvalues_and_annotate(p_values)
        return ax

In [ ]:
def super_boxplot_helper_singleplot(
    data_df,
    group_avg_df,
    ax,
    x_value,
    y_value,
    title,
    plate_col_name,
    pairs=None,
    order=None,
    annotate=False,
    pallete="Set2",
    test=None,
    shapiro=True,
    show_test_on_plot=False,
    ylim=None,
    context="talk",
    font_scale=1.2,
    p_correction="bonferroni",
    annotation_location="inside",
    limit_whiskers=True,
    min_percentile=5,
    max_percentile=95,
    annotation_line_offset=0.0,
    annotation_line_offset_to_group=0.005,
    annotation_text_offset=0.15,
    annotation_line_height=0.015,
):
    group_avg_df = group_avg_df.copy()
    if pairs is None:
        pairs = getpairs(data_df, x_value, order=order)
    if pallete is None:
        pallete = get_hard_code_plate_colours(group_avg_df)

    # Optionally trim extreme values so whiskers/axis are not dominated by outliers.
    plot_data_df = data_df.copy()
    plot_group_avg_df = group_avg_df.copy()

    if limit_whiskers:
        whiskers = (min_percentile, max_percentile)
    else:
        whiskers = (0, 100)
    
    sns.set_theme(style="ticks")
    sns.set_context(context=context, font_scale=font_scale)
    
    sns.boxplot(
        data=plot_data_df,
        x=x_value,
        y=y_value,
        order=order,
        color="gainsboro",
        width=0.6,
        fliersize=0,
        linewidth=1.5,
        whis=whiskers,
        showmeans=True,
        meanline=True,
        meanprops={"color": "#111111B2", "ls": "-", "lw": 3.0},
        medianprops={"color": "#555555", "ls": "-", "lw": 2.0},
        ax=ax,
    )
    sns.swarmplot(
        data=group_avg_df,
        x=x_value,
        y=y_value,
        hue=plate_col_name,
        order=order,
        palette=pallete,
        size=10,
        edgecolor="k",
        linewidth=1,
        dodge=False,
        ax=ax,
    )

    ax.set_title(title)

    if ylim is not None:
        if isinstance(ylim, (int, float)):
            ylim = (None, float(ylim))
        if len(ylim) == 2:
            ax.set_ylim(ylim)

    if annotate and test is not None:
        group_avg_pivot_table = average_groups_pivot(
            plot_group_avg_df, x_value, y_value, plate_col_name
        )
        try:
            print(f"Annotating with statistical test: {test}")
            ax = annotate_pairs_with_calculated_pvalues(
                ax,
                plot_group_avg_df,
                group_avg_pivot_table,
                x_value,
                y_value,
                plate_col_name=plate_col_name,
                test_name=test,
                pairs=pairs,
                order=order,
                plot_type="boxplot",
                show_test_name=show_test_on_plot,
                annotation_location=annotation_location,
                p_correction=p_correction,
                line_offset=annotation_line_offset,
                line_offset_to_group=annotation_line_offset_to_group,
                text_offset=annotation_text_offset,
                line_height=annotation_line_height,
            )
        except ValueError as e:
            print(f"Error annotating with statistical test: {e}")
        if shapiro:
            ax = annotate_legend_with_shapiro(ax, plot_group_avg_df, plate_col_name)

    return ax


def OLD_make_superboxplot_with_annotation(
    data_df,
    x_value,
    y_value,
    plate_col_name="PlateNumber",
    pairs=None,
    order=None,
    annotate=False,
    test="tukey",
    ytitle=None,
    xtitle=None,
    ylim=None,
    pallete=None,
    figsize=(10, 9),
    context="talk",
    dpi=300,
    p_correction="bonferroni",
    annotation_location="inside",
    show_plot=True,
    truncate_outliers=True,
    min_percentile=5,
    max_percentile=95,
    annotation_line_offset=0.0,
    annotation_line_offset_to_group=0.005,
    annotation_text_offset=0.15,
    annotation_line_height=0.015,
):
    plot_df = data_df.copy()
    # Set values to defaults if a parameter is not loaded for order or y axis parameters
    if ytitle is None:
        ytitle = y_value.replace("_", " ")
    if ylim is None:
        ylim = (0, 100)
    if order is None:
        order = data_df[x_value].dropna().unique().tolist()
    if xtitle is None:
        xtitle = x_value
    if pairs is None:
        pairs = getpairs(plot_df, x_value, order)
    print(pairs)

    # trim extreme values so whiskers are not waaay too long
    if truncate_outliers:
        ymin = np.nanpercentile(plot_df[y_value], min_percentile)
        ymax = np.nanpercentile(plot_df[y_value], max_percentile)
        print(
            f"Truncating {y_value} to percentiles "
            f"{min_percentile}-{max_percentile}: ({ymin}, {ymax})"
        )
        plot_df = plot_df[(plot_df[y_value] >= ymin) & (plot_df[y_value] <= ymax)]

    if x_value == "AllGroups":
        order = get_all_group_order()

    group_avg_df = average_groups_by_plate(
        plot_df, x_value=x_value, y_value=y_value, plates=plate_col_name
    )

    group_avg_df_shapiro = apply_shapiro_wilk_test_to_df(
        group_avg_df, feature_meas=y_value
    )

    group_avg_df_pivot = average_groups_pivot(
        group_avg_df=group_avg_df_shapiro,
        x_value=x_value,
        y_value=y_value,
        plate_col_name=plate_col_name,
    )

    sns.set_theme(style="ticks")
    plt.figure(figsize=figsize)
    sns.set_context(context, font_scale=0.8)
    if pallete is None:
        pallete = get_hard_code_plate_colours(plot_df)

    ax = sns.boxplot(
        data=plot_df,
        x=x_value,
        y=y_value,
        order=order,
        color="gainsboro",
        showmeans=True,
        meanline=True,
        showfliers=False,
        width=0.6,
        fliersize=0,
        linewidth=1.5,
        meanprops={"color": "#111111", "ls": "-", "lw": 3.0},
        medianprops={"color": "#555555", "ls": "-", "lw": 2.0},
    )

    sns.swarmplot(
        data=group_avg_df,
        x=x_value,
        y=y_value,
        hue=plate_col_name,
        order=order,
        palette=pallete,
        size=12,
        edgecolor="k",
        linewidth=1,
        dodge=False,
        ax=ax,
    )

    if ax.legend_ is not None:
        ax = annotate_legend_with_shapiro(ax, group_avg_df_shapiro, plate_col_name)

    sns.despine()
    plt.tight_layout()
    plt.xlabel(xtitle)
    plt.ylabel(ytitle)
    plt.ylim(ylim)
    if annotate and test is not None:
        try:
            print(test)
            ax = annotate_pairs_with_calculated_pvalues(
                ax,
                group_avg_df_shapiro,
                group_avg_df_pivot,
                x_value,
                y_value,
                plate_col_name=plate_col_name,
                test_name=test,
                order=order,
                annotation_location=annotation_location,
                pairs=pairs,
                plot_type="boxplot",
                p_correction=p_correction,
                line_offset=annotation_line_offset,
                line_offset_to_group=annotation_line_offset_to_group,
                text_offset=annotation_text_offset,
                line_height=annotation_line_height,
            )
        except ValueError as e:
            print(f"Error annotating with statistical test: {e}")

    plt.savefig(xtitle + "_" + y_value + "_superboxplot.png", dpi=dpi)
    if show_plot:
        plt.show()

In [ ]:
def single_feature_super_boxplot(
    data_df,
    x_value="AllGroups",
    y_value="AreaShape_Area",
    plate_col_name="PlateNumber",
    out_dir=Path("plots/"),
    xtitle=None,
    ytitle=None,
    order=None,
    legend=True,
    annotate=False,
    test="tukey_v3",
    ylim=None,
    shapiro=False,
    show=True,
    context="poster",
    font_scale=1.2,
    figsize=(8, 6),
    limit_whiskers=True,
    truncate_outliers=False,
    min_percentile=5,
    max_percentile=95,
    pallete="tab10",
    p_correction="bonf",
    annotation_location="inside",
    annotation_line_offset=0.0,
    annotation_line_offset_to_group=0.005,
    annotation_text_offset=0.005,
    annotation_line_height=0.005,
):
    """Make a superplot to do multiple comparisons for a feature between different conditions."""

    if order is None:
        order = get_all_group_order()

    # Validate percentiles.
    min_percentile = float(min_percentile)
    max_percentile = float(max_percentile)
    if not (0 <= min_percentile < max_percentile <= 100):
        raise ValueError(
            "truncate_min_percentile and truncate_max_percentile must satisfy "
            "0 <= min < max <= 100"
        )

    df_sorted = data_df.sort_values(
        by=[x_value], key=lambda x: x.map(allgroups_sort_key)
    ).reset_index(drop=True)
    feature_df = df_sorted[[x_value, y_value, plate_col_name]].copy()

    # if truncate_outliers is True, filter the feature_df to only include values within the specified percentiles
    if truncate_outliers:
        ymin = np.nanpercentile(feature_df[y_value], min_percentile)
        ymax = np.nanpercentile(feature_df[y_value], max_percentile)
        print(
            f"Truncating {y_value} to percentiles "
            f"{min_percentile}-{max_percentile}: ({ymin}, {ymax})"
        )
        feature_df = feature_df[
            (feature_df[y_value] >= ymin) & (feature_df[y_value] <= ymax)
        ]

    # group by plate and condition
    group_avg_df = feature_df.groupby([x_value, plate_col_name], as_index=False).mean()

    # sort the group avg df by the "AllGroups" order
    group_avg_df_sorted = group_avg_df.sort_values(
        by=[x_value], key=lambda x: x.map(allgroups_sort_key)
    ).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=figsize)

    # Calculate an adaptive y-limit from truncated data when not explicitly supplied.
    if ylim is None:
        y_values = feature_df[y_value].dropna()
        if not y_values.empty:
            y_low = np.nanpercentile(y_values, min_percentile)
            y_high = np.nanpercentile(y_values, max_percentile)
            if np.isfinite(y_low) and np.isfinite(y_high) and y_high > y_low:
                if truncate_outliers:
                    padding = (y_high - y_low) * 0.05
                    ylim = (y_low - padding, y_high + padding)
                # When not truncating, use a larger padding for the upper limit to avoid chopping off the top of the boxplot and annotations.
                elif np.isfinite(y_high) and y_high > 0:
                    padding_low = (y_high - y_low) * 0.05
                    y_high = np.nanpercentile(y_values, 99.75)
                    padding_high = (y_high - y_low) * 0.05
                    ylim = (y_low - padding_low, y_high * 1.05 + padding_high)
                else:
                    ylim = (None, None)
        else:
            ylim = (None, None)
    elif isinstance(ylim, (int, float)):
        ylim = (None, float(ylim))

    pairs = getpairs(feature_df, x_value, order=order)
    print(pairs)

    # Draw a super boxplot + plate-level points with optional percentile truncation.
    ax = super_boxplot_helper_singleplot(
        feature_df,
        group_avg_df_sorted,
        ax,
        x_value,
        y_value,
        title=" ",
        plate_col_name=plate_col_name,
        pairs=pairs,
        order=order,
        annotate=annotate,
        test=test,
        shapiro=False,
        pallete=pallete,
        p_correction=p_correction,
        annotation_location=annotation_location,
        show_test_on_plot=False,
        ylim=ylim,
        context=context,
        font_scale=font_scale,
        limit_whiskers=limit_whiskers,
        min_percentile=min_percentile,
        max_percentile=max_percentile,
        annotation_line_offset=annotation_line_offset,
        annotation_line_offset_to_group=annotation_line_offset_to_group,
        annotation_text_offset=annotation_text_offset,
        annotation_line_height=annotation_line_height,
    )

    if legend:
        ax.legend(
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=True,
            title=plate_col_name,
        )
        if shapiro:
            group_avg_df_shapiro = apply_shapiro_wilk_test_to_df(
                group_avg_df_sorted,
                feature_meas=y_value,
                plate_col_name="PlateNumber",
                alpha=0.05,
            )
            ax = annotate_legend_with_shapiro(ax, group_avg_df_shapiro, plate_col_name)
    else:
        if ax.legend_ is not None:
            ax.legend_.remove()

    if ytitle is not None:
        ax.set_ylabel(ytitle)
    else:
        ax.set_ylabel(y_value.replace("_", " "))

    if xtitle is not None:
        ax.set_xlabel(xtitle)

    # Set y-limits safely.
    if ylim is not None and len(ylim) == 2:
        print(f"Using y-limits: {ylim}")
        ax.set_ylim(ylim)

    plt.tight_layout()
    sns.despine()
    plt.savefig(os.path.join(out_dir, f"{y_value}_{test}.png"))
    if show:
        plt.show()

## Make a plot for a single feature

In [ ]:
order = get_all_group_order()
feature_meas = "AreaShape_Area"  # "AreaShape_Area"#Mean_Lysosomes_Distance_Centroid_Nuclei_PerCell_Area" #Mean_Lysosomes_DiameterRatio_PerCell"
ylabel = None  # "Mitochondrial Density Per Cell (relative to youngest passage)"#None#"Mitochondria per cell"
xlabel = "Age Groups"
group = "AllGroups"

pairs = getpairs(final_filtered_df_with_ijskeleton, group, order)

data_df = extrafeatures_filtered_cell_df_mitolyso.copy()

# Map each plate to a color and hard code that shit
colour_dict = get_hard_code_plate_colours(data_df)
pallete = colour_dict  # "pastel"

plot_dir = "plots/notnorm"
os.makedirs(plot_dir, exist_ok=True)

figsize = (12, 10)

single_feature_super_boxplot(
    data_df,
    x_value=group,
    y_value=feature_meas,
    plate_col_name="PlateNumber",
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=False,
    order=order,
    test="tukey_v3",
    truncate_outliers=False,
    limit_whiskers=True,
    legend=False,
    context="poster",
    font_scale=1,
    figsize=figsize,
    pallete=pallete,
    p_correction="fdr_bh",
    annotation_location="outside",
    show=True,
    # truncate_outliers=True
)

# NOTE: R5 has smallest cells in p23-25, which is also highest mito density. Also note plate 3 has lysosome staining abnormalities in the first two cols

In [ ]:
# Lineage Version
order = get_all_group_order()
feature_meas = "AreaShape_Area"  # Mean_Lysosomes_Distance_Centroid_Nuclei_PerCell_Area" #Mean_Lysosomes_DiameterRatio_PerCell"
ylabel = None  # "Mitochondrial Density Per Cell (relative to youngest passage)"#None#"Mitochondria per cell"
xlabel = "Age Groups"
group = "AllGroups"
stat_grouping = "Lineage"

pairs = getpairs(combined_cell_df_mitolyso, group, order)

this_df = extrafeatures_filtered_cell_df_mitolyso.copy()
data_df = this_df

# Uncomment this if the fix to add "doxo_" to the doxo lineages is not set
# no_drug_values = {None,"", "None", "none", "NoDrug", "Control", "control"}

# drug_clean = data_df["Drug"].astype(str).str.strip()
# data_df["Lineage"] = data_df["Lineage"].where(
#     data_df["Drug"].isna() | drug_clean.isin(no_drug_values),
#     drug_clean + "_" + data_df["Lineage"].astype(str),
# )

data_df["Lineage"] = data_df["Lineage"].replace(
    "LIN3-0A-b2-s2-ss1-ss1", "LIN3-0A-b2-s2-ss1-sss1"
)
#     data_df["Lineage"].equals("LIN3-0A-b2-s2-ss1-ss1"),
#     data_df["Lineage"].equals("LIN3-0A-b2-s2-ss1-sss1"),
# )


def get_hard_code_lineage_colours(
    df, lineage_col_name="Lineage", plate_col_name="PlateNumber"
):
    """
    Returns a dictionary mapping each unique lineage number to a hard-coded color.
    This ensures color consistency for each PlateNumber in seaborn/matplotlib plots.
    """
    # Get colour codes from my csv and then convert it into a dict in the {key: #hexcode} format
    colour_codes = pd.read_csv("proliferation_growth_curves/Lineage_only_hexcolour.csv")
    hard_pallete_dict = dict(
        zip(colour_codes[lineage_col_name], colour_codes["Colour"])
    )

    # print(hard_pallete_dict)
    unique_lineages = sorted(df[lineage_col_name].drop_duplicates())
    # display(unique_lineages)

    # If more lineages than colors, fill in missing colours by using a default seaborn color_palette
    for lineage in unique_lineages:
        if lineage not in hard_pallete_dict:
            extra_colours = sns.color_palette("tab20", len(unique_lineages)).as_hex()
            hard_pallete_dict = {
                lineage: extra_colours[i] for i, lineage in enumerate(unique_lineages)
            }
    return hard_pallete_dict


lindict = {
    "LIN1-0B-b1": "#8000FF",
    "LIN2-0A-b1": "#304CC9",
    "LIN3-0A-b2": "#02FD68",
    "LIN4-0B-b2": "#FFF604",
    "LIN5-0B-b3": "#FFDC00",
    "LIN6-0C-b1": "#FFA000",
    "LIN2-0A-b1-s1": "#87A5FF",
    "LIN3-0A-b2-s1": "#01FDBD",
    "LIN4-0B-b2A-s1": "#E0E102",
    "LIN5-0B-b3-s1": "#958300",
    "LIN6-0C-b1-s1": "#FFB479",
    "LIN3-0A-b2-s2-ss1": "#87FE02",
    "LIN7-0C-b3": "#FF7100",
    "LIN8-0B-b2B-s1": "#FF3200",
    "LIN9-0C-b2-s1": "#FF768A",
    "LIN10-0C-b4": "#A770A3",
    "Doxo_LIN13-0C-b2-s3": "#808080",
    "LIN3-0A-b2-s1-ss1": "#00F0FF",
    "LIN3-0A-b2-s1-ss1-sss1": "#00D7FF",
    "LIN3-0A-b2-s1-ss2": "#019BFF",
    "LIN3-0A-b2-s2-ss1-sss1": "#D5FF93",
    "LIN7-0C-b3-s1": "#FF8F85",
    "LIN8-0B-b2B-s1-ss1": "#D2372C",
    "LIN8-0B-b2B-s1-ss2": "#8C1600",
    "LIN6-0C-b1-s2": "#FFD7AD",
    "LIN11-0C-b5": "#FF2E9D",
    "Doxo_LIN11-0C-b5": "#780A78",
    "LIN12-0C-b2-s2": "#FF23FF",
    "Doxo_LIN12-0C-b2-s2": "#BE12D9",
    "LIN2-0A-b1-s1-ss1": "#A8DAFC",
    "LIN7-0C-b3-s2": "#D0A38C",
    "LIN11-0C-b5-s1": "#FFBFFF",
}
# Map each plate to a color and hard code that shit
colour_dict = get_hard_code_lineage_colours(data_df)
pallete = lindict

remove_outliers = False
reps_to_exclude = []
plot_dir = "plots/lineage"
os.makedirs(plot_dir, exist_ok=True)

figsize = (11, 18)

single_feature_super_splitviolinplot(
    data_df,
    x_value=group,
    y_value=feature_meas,
    plate_col_name=stat_grouping,
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=True,
    order=order,
    test="tukey_v3",
    reps_to_exclude=reps_to_exclude,
    show_hist=False,  # True,
    remove_outliers=False,
    truncate_outliers=True,
    legend=True,
    context="talk",
    figsize=figsize,
    pallete=pallete,
    shapiro=False,
    # p_correction="fdr_bh",
    # truncate_outliers=True
)

# NOTE: R5 has smallest cells in p23-25, which is also highest mito density


## Make multiple plots as defined

In [ ]:
# take feature : label pairs to be used for plots from csv
features_df = pd.read_csv("CP_features_for_plots.csv")
display(features_df[features_df["feature"] == "AreaShape_Area"])

data_df = extrafeatures_filtered_cell_df_mitolyso.copy()
# Use in plotting loops with guaranteed alignment:


def make_feature_plots_from_csv(
    data_df,
    features_df,
    xlabel="Age Groups",
    group="AllGroups",
    analysis_mode="Plates",
    norm=False,
    order=None,
    figsize=(10, 10),
    truncate_outliers=False,
    annotate_pval=True,
    test="tukey_v3",
    font_scale=1.0,
    annotation_location="inside",
    context="poster",
    show_legend=False,
):
    features_for_plots = features_df.to_dict("records")

    feature_df_cols = [item["feature"] for item in features_for_plots]
    feature_labels = [item["label"] for item in features_for_plots]

    if order is None:
        order = get_all_group_order()

    # Set stats variables and color palette based on analysis mode
    if analysis_mode == "Lineages":
        stat_grouping = "Lineage"
        plot_dir = Path("plots/lineages")
        show_legend = True
        colour_dict = get_hard_code_lineage_colours(data_df)
    elif analysis_mode == "Plates":
        stat_grouping = "PlateNumber"
        colour_dict = get_hard_code_plate_colours(data_df)
        if norm:
            data_df = norm_combined_cell_df_mitolyso.copy()
            feature_labels = [
                f"{name} (normalized to youngest group)" for name in feature_labels
            ]
            plot_dir = Path("plots/norm")
        else:
            data_df.copy()
            plot_dir = Path("plots/notnorm")
            
    else:
        raise ValueError(
            f"Invalid analysis mode: {analysis_mode}. Use 'Lineages' or 'Plates'."
        )
    pallete = colour_dict  # "pastel"

    if annotate_pval is False:
        test = None
        plot_dir = Path(plot_dir, "simplified")
    os.makedirs(plot_dir, exist_ok=True)
    for i, feature in enumerate(feature_df_cols):
        ylabel = feature_labels[i]

        single_feature_super_boxplot(
            data_df,
            x_value=group,
            y_value=feature,
            plate_col_name=stat_grouping,
            xtitle=xlabel,
            ytitle=ylabel,
            out_dir=plot_dir,
            annotate=annotate_pval,
            order=order,
            test=test,
            truncate_outliers=truncate_outliers,
            legend=show_legend,
            context=context,
            figsize=figsize,
            pallete=pallete,
            p_correction="fdr_bh",
            shapiro=False,
            font_scale=font_scale,
            show=False,
            annotation_location=annotation_location,
        )
        # single_feature_super_splitviolinplot(
        #     data_df,
        #     x_value=group,
        #     y_value=feature,
        #     plate_col_name=stat_grouping,
        #     xtitle=xlabel,
        #     ytitle=ylabel,
        #     out_dir=plot_dir,
        #     annotate=annotate_pval,
        #     order=order,
        #     test=test,
        #     reps_to_exclude=reps_to_exclude,
        #     show_hist=False,  # True,
        #     remove_outliers=remove_outliers,
        #     rm_outliers_method=rm_outliers_method,
        #     truncate_outliers=truncate_outliers,
        #     legend=show_legend,
        #     context=context,
        #     figsize=figsize,
        #     pallete=pallete,
        #     p_correction="fdr_bh",
        #     shapiro=False,
        #     font_scale=font_scale,
        #     show=False,
        #     annotation_location=annotation_location,
        # )


make_feature_plots_from_csv(
    data_df,
    features_df,
    norm=False,
    analysis_mode="Plates",
    truncate_outliers=False,
    figsize=(12, 14),
    annotate_pval=True,
    test="tukey_v3",
    font_scale=1.0,
    annotation_location="outside",
    context="poster",
)

make_feature_plots_from_csv(
    data_df,
    features_df,
    norm=False,
    analysis_mode="Plates",
    truncate_outliers=False,
    figsize=(12, 10),
    annotate_pval=False,
    test=None,
    font_scale=1.0,
    annotation_location="outside",
    context="poster",
)

make_feature_plots_from_csv(
    data_df,
    features_df,
    norm=True,
    analysis_mode="Plates",
    truncate_outliers=False,
    figsize=(12, 14),
    annotate_pval=True,
    test="dunn",
    font_scale=1.0,
    annotation_location="outside",
    context="poster",
)


In [ ]:

make_feature_plots_from_csv(
    data_df,
    features_df,
    norm=True,
    analysis_mode="Plates",
    truncate_outliers=False,
    figsize=(12, 16),
    annotate_pval=True,
    test=None,
    font_scale=1.0,
    annotation_location="outside",
    context="poster",
)

## Attempting PCA

## Summary Stats

In [ ]:
summary_outpath = "postprocessed_summary_stats"
os.makedirs(summary_outpath, exist_ok=True)

feature_cols = [
    "AreaShape_Area",
    "Nuclei_AreaShape_Area",
    "Cell_Nuclei_Area_Ratio",
    "Nuclei_Intensity_MedianIntensity_DAPI_MAX_WellNormalized",
    "Intensity_MeanIntensity_LAMP1_MAX_WellNormalized",
    "Intensity_MeanIntensity_MitoTracker_MAX_WellNormalized",
    "Neighbors_NumberOfNeighbors_5",
    "Neighbors_PercentTouching_5",
    "Nuclei_Neighbors_NumberOfNeighbors_1",
    "Nuclei_Neighbors_PercentTouching_1",
    "Children_Lysosomes_Count",
    "Children_Mitochondria_Count",
]
include_cols = ["Number_Object_Number"] + feature_cols
print(include_cols)


def make_summary_stats_for_df_and_feature(
    df,
    x_value,
    feature,
    summary_outpath,
    df_tag="original",
    plate_col_name="PlateNumber",
    feature_name="area",
    group_name="passage_group",
    include_cols=[],
    inculded_percentiles=[
        0.01,
        0.025,
        0.05,
        0.1,
        0.25,
        0.5,
        0.75,
        0.9,
        0.95,
        0.975,
        0.99,
    ],
):
    from pathlib import Path

    try:
        table_csvname = f"{df_tag}_total_combined_stats.csv"
        feature_csvname = f"{df_tag}_{feature_name}_by_{group_name}_stats.csv"
        agg_feature_csvname = f"{df_tag}_agg_{feature_name}_by_{group_name}_stats.csv"

        subfolder_name = f"{df_tag}_{feature_name}_summary_stats"
        parent_folder = Path(summary_outpath, subfolder_name)
        parent_folder.mkdir(exist_ok=True)

        if not include_cols:
            df_to_summarize = df
        else:
            df_to_summarize = df[include_cols]
        df_to_summarize.describe(percentiles=inculded_percentiles).to_csv(
            os.path.join(summary_outpath, table_csvname)
        )
        group_averages = df.groupby(
            [x_value, plate_col_name], as_index=False, observed=True
        )[feature]
        # Reset the index to get a clean DataFrame
        # average_df = group_averages.reset_index()
        avg_summary = group_averages.describe(percentiles=inculded_percentiles)
        avg_summary_sorted = avg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        avg_summary_sorted.to_csv(
            os.path.join(summary_outpath, subfolder_name, feature_csvname)
        )

        # do the agg by passage group only
        group_averages_agg = df.groupby([x_value], as_index=False, observed=True)[
            feature
        ]
        avg_agg_summary = group_averages_agg.describe(percentiles=inculded_percentiles)
        avg_agg_summary_sorted = avg_agg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        avg_agg_summary_sorted.T.to_csv(
            os.path.join(summary_outpath, subfolder_name, agg_feature_csvname)
        )
        print(
            f"saved files {(table_csvname, feature_csvname, agg_feature_csvname)} to {summary_outpath}"
        )
        return True
    except ValueError as e:
        print(f"Could not make summary stats: {e}")
        return False


this_df = combined_cell_df_mitolyso #extrafeatures_filtered_cell_df_mitolyso.copy()


for feature in feature_cols:
    df_sorted = this_df.sort_values(
        by=["AllGroups"], key=lambda x: x.map(passage_groups_sort_key)
    ).reset_index(drop=True)
    make_summary_stats_for_df_and_feature(
        df_sorted,
        "AllGroups",
        feature,
        summary_outpath,
        df_tag="original",
        feature_name=feature,
        include_cols=include_cols,
    )
    # seaborn_ridgeplot(
    #     df_sorted,
    #     value_col=feature,
    #     group_col="AllGroups",
    #     palette="Set2",
    #     save=True,
    #     out_dir=summary_outpath,
    #     show_percentiles=True,
    #     truncate_outliers=True,
    # )


### To export the normalized csv:


In [ ]:
combined_cell_df_mitolyso_borders_excluded = pd.read_csv(
    os.path.join(csvpath, filename_borders_excluded)
)
filtered_borders_excluded = apply_all_filters(
    combined_cell_df_mitolyso_borders_excluded
)

for feature in feature_cols:
    make_summary_stats_for_df_and_feature(
        filtered_borders_excluded,
        "AllGroups",
        feature,
        summary_outpath,
        df_tag="borders_excluded",
        feature_name=feature,
        include_cols=include_cols,
    )


In [ ]:
preprocessed_df = extrafeatures_filtered_cell_df_mitolyso.copy()

preprocessed_df.to_csv(
    os.path.join(csvpath, "CellProfiler_features_preprocessed.csv"), index=False
)

norm_cell_df_mitolyso.to_csv(
    os.path.join(csvpath, "Norm_CellProfiler_features_preprocessed.csv"),
    index=False,
)